# Trader Performance vs Market Sentiment
### Hyperliquid Historical Trades x Bitcoin Fear/Greed Index

**Author:** Data Science Intern Assignment — Primetrade.ai
**Data:** 211,224 trades across 32 accounts (May 2023 – May 2025) and the daily Bitcoin Fear/Greed Index.

This notebook covers:
- **Part A** — Data loading, cleaning, alignment, and metric construction
- **Part B** — Sentiment vs performance/behavior analysis, and trader segmentation
- **Part C** — Actionable strategy recommendations
- **Bonus** — A simple next-day profitability classifier + KMeans trader archetypes

Run `pip install -r requirements.txt` then run cells top to bottom, or run the standalone scripts in `scripts/` in order (`01_prepare.py`, `02_analysis.py`, `03_bonus.py`).


## Part A — Data Preparation

In [ ]:
"""
Part A - Data preparation
Loads trader data + fear/greed index, cleans, aligns by date,
and builds the core daily/account-level metrics used in the analysis.
"""
import pandas as pd
import numpy as np

pd.set_option('display.width', 140)

# ---------------------------------------------------------------
# 1. Load
# ---------------------------------------------------------------
trades = pd.read_csv('data/historical_data__2_.csv')
fg = pd.read_csv('data/fear_greed_index__2_.csv')

print("=== RAW SHAPES ===")
print("trades:", trades.shape)
print("fear_greed:", fg.shape)

print("\n=== MISSING VALUES ===")
print("trades:\n", trades.isna().sum()[trades.isna().sum() > 0])
print("fear_greed:\n", fg.isna().sum()[fg.isna().sum() > 0])

print("\n=== DUPLICATES ===")
print("trades duplicated rows:", trades.duplicated().sum())
print("fear_greed duplicated rows:", fg.duplicated().sum())

# ---------------------------------------------------------------
# 2. Clean + parse dates
# ---------------------------------------------------------------
trades['datetime'] = pd.to_datetime(trades['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce')
trades['date'] = trades['datetime'].dt.date
trades['date'] = pd.to_datetime(trades['date'])

fg['date'] = pd.to_datetime(fg['date'])
fg = fg[['date', 'value', 'classification']].drop_duplicates(subset='date').sort_values('date')

# Collapse 5-way classification into a simple Fear/Greed/Neutral bucket
def bucket(c):
    if c in ('Fear', 'Extreme Fear'):
        return 'Fear'
    if c in ('Greed', 'Extreme Greed'):
        return 'Greed'
    return 'Neutral'

fg['sentiment_bucket'] = fg['classification'].apply(bucket)

print("\n=== TRADE DATE RANGE ===")
print(trades['date'].min(), '->', trades['date'].max())
print("=== SENTIMENT DATE RANGE ===")
print(fg['date'].min(), '->', fg['date'].max())
print("unique accounts:", trades['Account'].nunique())
print("unique coins:", trades['Coin'].nunique())

# ---------------------------------------------------------------
# 3. Row-level flags
# ---------------------------------------------------------------
# A "closing" trade is one that realizes PnL (Closed PnL != 0 OR direction contains Close)
trades['is_close'] = trades['Closed PnL'] != 0
trades['is_win'] = trades['Closed PnL'] > 0
trades['is_long'] = trades['Direction'].str.contains('Long', case=False, na=False) | (trades['Side'] == 'BUY')
trades['is_short'] = trades['Direction'].str.contains('Short', case=False, na=False) | (trades['Side'] == 'SELL')

# ---------------------------------------------------------------
# 4. Merge trades with sentiment on date
# ---------------------------------------------------------------
merged = trades.merge(fg[['date', 'value', 'classification', 'sentiment_bucket']], on='date', how='left')
print("\n=== TRADES WITHOUT SENTIMENT MATCH ===")
print(merged['sentiment_bucket'].isna().sum(), "/", len(merged))

merged = merged.dropna(subset=['sentiment_bucket'])

merged.to_csv('outputs/merged_trades.csv', index=False)
print("\nSaved outputs/merged_trades.csv", merged.shape)

# ---------------------------------------------------------------
# 5. Daily x Account metrics  (the core analysis table)
# ---------------------------------------------------------------
daily_acct = merged.groupby(['date', 'Account']).agg(
    trades_count=('Trade ID', 'count'),
    total_pnl=('Closed PnL', 'sum'),
    closing_trades=('is_close', 'sum'),
    wins=('is_win', 'sum'),
    avg_trade_size_usd=('Size USD', 'mean'),
    total_volume_usd=('Size USD', 'sum'),
    long_trades=('is_long', 'sum'),
    short_trades=('is_short', 'sum'),
    total_fees=('Fee', 'sum'),
).reset_index()

daily_acct['win_rate'] = np.where(daily_acct['closing_trades'] > 0,
                                   daily_acct['wins'] / daily_acct['closing_trades'], np.nan)
daily_acct['long_short_ratio'] = np.where(daily_acct['short_trades'] > 0,
                                           daily_acct['long_trades'] / daily_acct['short_trades'],
                                           np.nan)

daily_acct = daily_acct.merge(fg[['date', 'value', 'classification', 'sentiment_bucket']], on='date', how='left')
daily_acct.to_csv('outputs/daily_account_metrics.csv', index=False)
print("Saved outputs/daily_account_metrics.csv", daily_acct.shape)

# ---------------------------------------------------------------
# 6. Daily x Market (all accounts pooled) metrics
# ---------------------------------------------------------------
daily_mkt = merged.groupby('date').agg(
    trades_count=('Trade ID', 'count'),
    total_pnl=('Closed PnL', 'sum'),
    closing_trades=('is_close', 'sum'),
    wins=('is_win', 'sum'),
    avg_trade_size_usd=('Size USD', 'mean'),
    total_volume_usd=('Size USD', 'sum'),
    long_trades=('is_long', 'sum'),
    short_trades=('is_short', 'sum'),
    active_accounts=('Account', 'nunique'),
).reset_index()
daily_mkt['win_rate'] = daily_mkt['wins'] / daily_mkt['closing_trades']
daily_mkt['long_short_ratio'] = daily_mkt['long_trades'] / daily_mkt['short_trades']
daily_mkt = daily_mkt.merge(fg[['date', 'value', 'classification', 'sentiment_bucket']], on='date', how='left')

# simple drawdown proxy on cumulative pooled PnL
daily_mkt = daily_mkt.sort_values('date')
daily_mkt['cum_pnl'] = daily_mkt['total_pnl'].cumsum()
daily_mkt['running_max'] = daily_mkt['cum_pnl'].cummax()
daily_mkt['drawdown'] = daily_mkt['cum_pnl'] - daily_mkt['running_max']

daily_mkt.to_csv('outputs/daily_market_metrics.csv', index=False)
print("Saved outputs/daily_market_metrics.csv", daily_mkt.shape)

# ---------------------------------------------------------------
# 7. Account-level summary (for segmentation)
# ---------------------------------------------------------------
acct_summary = daily_acct.groupby('Account').agg(
    active_days=('date', 'nunique'),
    total_trades=('trades_count', 'sum'),
    avg_trades_per_day=('trades_count', 'mean'),
    total_pnl=('total_pnl', 'sum'),
    avg_daily_pnl=('total_pnl', 'mean'),
    pnl_std=('total_pnl', 'std'),
    avg_trade_size_usd=('avg_trade_size_usd', 'mean'),
    avg_win_rate=('win_rate', 'mean'),
    win_rate_std=('win_rate', 'std'),
).reset_index()
acct_summary['pnl_consistency'] = acct_summary['avg_daily_pnl'] / acct_summary['pnl_std'].replace(0, np.nan)
acct_summary.to_csv('outputs/account_summary.csv', index=False)
print("Saved outputs/account_summary.csv", acct_summary.shape)

print("\nDONE - Part A complete")


### Part A notes

- **historical_data.csv**: 211,224 rows x 16 columns. Zero missing values, zero exact duplicate rows.
- **fear_greed_index.csv**: 2,644 rows x 4 columns. Zero missing values, zero duplicates.
- Trades span **2023-05-01 to 2025-05-01**; sentiment data covers 2018-2025, so full overlap exists.
- Only **6 of 211,224 trades** (0.003%) had no matching sentiment date and were dropped — negligible.
- The 5-way sentiment classification (`Extreme Fear`, `Fear`, `Neutral`, `Greed`, `Extreme Greed`) was
  collapsed into a 3-way bucket (`Fear`, `Neutral`, `Greed`) for cleaner comparison, since the extreme
  categories are sparse on a daily-account basis.
- **Leverage caveat**: the raw extract does not include an explicit margin/leverage field (no collateral
  or account-equity column), so "leverage distribution" from the brief is approximated using **trade
  notional size (Size USD)** as a proxy for risk-taking intensity. This is flagged throughout the analysis.


## Part B — Analysis

In [ ]:
"""
Part B - Analysis
Compares performance & behavior across Fear / Greed / Neutral days,
builds 3 trader segments, and runs simple significance checks.
"""
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

daily_acct = pd.read_csv('outputs/daily_account_metrics.csv', parse_dates=['date'])
daily_mkt = pd.read_csv('outputs/daily_market_metrics.csv', parse_dates=['date'])
acct_summary = pd.read_csv('outputs/account_summary.csv')
merged = pd.read_csv('outputs/merged_trades.csv', parse_dates=['date'])

order = ['Fear', 'Neutral', 'Greed']
palette = {'Fear': '#c0392b', 'Neutral': '#7f8c8d', 'Greed': '#27ae60'}

# =================================================================
# B1. Performance by sentiment (pooled market, and account-day level)
# =================================================================
print("### Daily pooled PnL by sentiment ###")
g = daily_mkt.groupby('sentiment_bucket')['total_pnl'].agg(['count', 'mean', 'median', 'std'])
print(g.reindex(order))

print("\n### Account-day win rate by sentiment ###")
g2 = daily_acct.groupby('sentiment_bucket')['win_rate'].agg(['count', 'mean', 'median'])
print(g2.reindex(order))

print("\n### Account-day PnL by sentiment ###")
g3 = daily_acct.groupby('sentiment_bucket')['total_pnl'].agg(['count', 'mean', 'median', 'std'])
print(g3.reindex(order))

# Mann-Whitney U test: Fear vs Greed, account-day PnL (non-normal, robust to outliers)
fear_pnl = daily_acct.loc[daily_acct.sentiment_bucket == 'Fear', 'total_pnl'].dropna()
greed_pnl = daily_acct.loc[daily_acct.sentiment_bucket == 'Greed', 'total_pnl'].dropna()
u_stat, p_val = stats.mannwhitneyu(fear_pnl, greed_pnl, alternative='two-sided')
print(f"\nMann-Whitney U (account-day PnL, Fear vs Greed): U={u_stat:.0f}, p={p_val:.4f}")

fear_wr = daily_acct.loc[daily_acct.sentiment_bucket == 'Fear', 'win_rate'].dropna()
greed_wr = daily_acct.loc[daily_acct.sentiment_bucket == 'Greed', 'win_rate'].dropna()
u_stat2, p_val2 = stats.mannwhitneyu(fear_wr, greed_wr, alternative='two-sided')
print(f"Mann-Whitney U (account-day win rate, Fear vs Greed): U={u_stat2:.0f}, p={p_val2:.4f}")

# Drawdown proxy: avg daily drawdown by sentiment (pooled)
print("\n### Drawdown proxy (pooled cumulative PnL) by sentiment ###")
print(daily_mkt.groupby('sentiment_bucket')['drawdown'].agg(['mean', 'min']).reindex(order))

# -----------------------------------------------------------------
# Chart 1: Distribution of account-day PnL by sentiment (boxplot, clipped)
# -----------------------------------------------------------------
plot_df = daily_acct[daily_acct['total_pnl'].abs() < daily_acct['total_pnl'].abs().quantile(0.95)]
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=plot_df, x='sentiment_bucket', y='total_pnl', order=order, palette=palette, ax=ax, showfliers=False)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Account-Day Closed PnL by Market Sentiment\n(outliers >95th pct trimmed for readability)')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Closed PnL (USD)')
plt.tight_layout()
plt.savefig('outputs/chart1_pnl_by_sentiment.png')
plt.close()

# -----------------------------------------------------------------
# Chart 2: Win rate by sentiment
# -----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 5))
wr_summary = daily_acct.groupby('sentiment_bucket')['win_rate'].mean().reindex(order)
bars = ax.bar(wr_summary.index, wr_summary.values, color=[palette[k] for k in wr_summary.index])
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005, f"{b.get_height():.1%}", ha='center')
ax.set_title('Average Account-Day Win Rate by Market Sentiment')
ax.set_ylabel('Win Rate')
ax.set_ylim(0, max(wr_summary.values) * 1.25)
plt.tight_layout()
plt.savefig('outputs/chart2_winrate_by_sentiment.png')
plt.close()

# =================================================================
# B2. Behavior by sentiment: trade frequency, size, long/short bias
# =================================================================
print("\n### Behavior by sentiment ###")
beh = daily_acct.groupby('sentiment_bucket').agg(
    avg_trades_per_acct_day=('trades_count', 'mean'),
    avg_trade_size_usd=('avg_trade_size_usd', 'mean'),
    avg_long_short_ratio=('long_short_ratio', 'mean'),
).reindex(order)
print(beh)

# long/short bias directly from row-level data
ls = merged.groupby('sentiment_bucket').apply(
    lambda d: pd.Series({
        'pct_long': d['is_long'].sum() / (d['is_long'].sum() + d['is_short'].sum()),
        'pct_short': d['is_short'].sum() / (d['is_long'].sum() + d['is_short'].sum()),
        'avg_size_usd': d['Size USD'].mean(),
        'trades': len(d)
    })
).reindex(order)
print("\n### Long/Short bias (row-level) ###")
print(ls)

# -----------------------------------------------------------------
# Chart 3: Trade frequency & avg trade size by sentiment (dual panel)
# -----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
beh['avg_trades_per_acct_day'].plot(kind='bar', ax=axes[0], color=[palette[k] for k in beh.index])
axes[0].set_title('Avg Trades per Account-Day')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

beh['avg_trade_size_usd'].plot(kind='bar', ax=axes[1], color=[palette[k] for k in beh.index])
axes[1].set_title('Avg Trade Size (USD)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('outputs/chart3_frequency_size_by_sentiment.png')
plt.close()

# -----------------------------------------------------------------
# Chart 4: Long vs Short share by sentiment (stacked bar)
# -----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 5))
ls[['pct_long', 'pct_short']].plot(kind='bar', stacked=True, ax=ax,
                                    color=['#2980b9', '#e67e22'])
ax.set_title('Long vs Short Trade Share by Sentiment')
ax.set_ylabel('Share of Trades')
ax.tick_params(axis='x', rotation=0)
ax.legend(['Long', 'Short'])
plt.tight_layout()
plt.savefig('outputs/chart4_long_short_by_sentiment.png')
plt.close()

# =================================================================
# B3. Segmentation
# =================================================================
# Segment 1: High vs Low trade-size traders
size_median = acct_summary['avg_trade_size_usd'].median()
acct_summary['size_segment'] = np.where(acct_summary['avg_trade_size_usd'] >= size_median, 'High Size', 'Low Size')

# Segment 2: Frequent vs Infrequent traders
freq_median = acct_summary['avg_trades_per_day'].median()
acct_summary['freq_segment'] = np.where(acct_summary['avg_trades_per_day'] >= freq_median, 'Frequent', 'Infrequent')

# Segment 3: Consistent winners vs inconsistent (based on win_rate mean & std)
wr_median = acct_summary['avg_win_rate'].median()
consistency_median = acct_summary['win_rate_std'].median()
def consistency_label(row):
    if row['avg_win_rate'] >= wr_median and row['win_rate_std'] <= consistency_median:
        return 'Consistent Winner'
    elif row['avg_win_rate'] < wr_median and row['win_rate_std'] > consistency_median:
        return 'Inconsistent'
    else:
        return 'Mixed'
acct_summary['consistency_segment'] = acct_summary.apply(consistency_label, axis=1)

acct_summary.to_csv('outputs/account_summary_segmented.csv', index=False)
print("\n### Segment sizes ###")
print(acct_summary['size_segment'].value_counts())
print(acct_summary['freq_segment'].value_counts())
print(acct_summary['consistency_segment'].value_counts())

# Merge segments back onto daily_acct for sentiment x segment analysis
seg_map = acct_summary.set_index('Account')[['size_segment', 'freq_segment', 'consistency_segment']]
daily_acct_seg = daily_acct.merge(seg_map, on='Account', how='left')
daily_acct_seg.to_csv('outputs/daily_account_segmented.csv', index=False)

print("\n### PnL by Size Segment x Sentiment ###")
print(daily_acct_seg.groupby(['size_segment', 'sentiment_bucket'])['total_pnl'].mean().unstack().reindex(columns=order))

print("\n### PnL by Freq Segment x Sentiment ###")
print(daily_acct_seg.groupby(['freq_segment', 'sentiment_bucket'])['total_pnl'].mean().unstack().reindex(columns=order))

print("\n### Win rate by Consistency Segment x Sentiment ###")
print(daily_acct_seg.groupby(['consistency_segment', 'sentiment_bucket'])['win_rate'].mean().unstack().reindex(columns=order))

# -----------------------------------------------------------------
# Chart 5: PnL by size segment x sentiment (grouped bar)
# -----------------------------------------------------------------
pivot = daily_acct_seg.groupby(['sentiment_bucket', 'size_segment'])['total_pnl'].mean().unstack()
pivot = pivot.reindex(order)
fig, ax = plt.subplots(figsize=(8, 5))
pivot.plot(kind='bar', ax=ax, color=['#8e44ad', '#f1c40f'])
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Avg Account-Day PnL: High vs Low Trade-Size Traders, by Sentiment')
ax.set_ylabel('Avg PnL (USD)')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('outputs/chart5_pnl_size_segment_sentiment.png')
plt.close()

# -----------------------------------------------------------------
# Chart 6: Trade frequency by freq segment x sentiment
# -----------------------------------------------------------------
pivot2 = daily_acct_seg.groupby(['sentiment_bucket', 'freq_segment'])['trades_count'].mean().unstack().reindex(order)
fig, ax = plt.subplots(figsize=(8, 5))
pivot2.plot(kind='bar', ax=ax, color=['#16a085', '#d35400'])
ax.set_title('Avg Trades per Account-Day: Frequent vs Infrequent Traders, by Sentiment')
ax.set_ylabel('Avg Trades / Day')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('outputs/chart6_freq_segment_sentiment.png')
plt.close()

print("\nDONE - Part B complete. Charts saved to outputs/.")


### Chart 1 — PnL distribution by sentiment
![chart1](../outputs/chart1_pnl_by_sentiment.png)

### Chart 2 — Win rate by sentiment
![chart2](../outputs/chart2_winrate_by_sentiment.png)

### Chart 3 — Trade frequency & size by sentiment
![chart3](../outputs/chart3_frequency_size_by_sentiment.png)

### Chart 4 — Long/short bias by sentiment
![chart4](../outputs/chart4_long_short_by_sentiment.png)

### Chart 5 — PnL: high vs low trade-size traders, by sentiment
![chart5](../outputs/chart5_pnl_size_segment_sentiment.png)

### Chart 6 — Trade frequency: frequent vs infrequent traders, by sentiment
![chart6](../outputs/chart6_freq_segment_sentiment.png)


### Part B findings — see the full write-up in `WRITEUP.md` for detailed interpretation.

Quick summary of what the numbers show:

1. **Fear days have the highest average PnL per account-day (~$5,185) vs Greed (~$4,144) and Neutral (~$3,439)**,
   but the difference is *not* statistically significant at the 5% level (Mann-Whitney U, p≈0.06 for PnL,
   p≈0.26 for win rate). Win rates are similar across regimes (~83-86%), so the PnL edge on Fear days is
   driven by trade sizing and volatility, not accuracy.
2. **Traders scale up on Fear days**: trades/account-day rises (105 vs 77 on Greed) and average trade size
   nearly doubles (~$7,182 vs ~$4,574). They also lean more long-biased (54% long) on Fear days than on
   Greed days (47% long, i.e. tilted short) — a mild contrarian pattern (buying fear, some profit-taking/short
   bias on greed).
3. **Segment interaction matters**: High trade-size traders earn far more on Fear days (~$9,540 avg) than
   Low trade-size traders (~$2,576) — the sentiment effect is concentrated in traders who already size up.
   Frequent traders show the same pattern (~$7,955 on Fear vs ~$2,525 for infrequent).
4. **Consistent-winner accounts hold a stable ~90%+ win rate across all sentiment regimes**, while
   inconsistent accounts sit around 75-78% regardless of sentiment — sentiment does not close the
   skill/consistency gap between trader types.


## Part C — Actionable Output

**Strategy idea 1 — Sentiment-scaled risk budget for high-conviction (large-size) traders.**
High trade-size accounts show a 3.7x PnL uplift on Fear days vs Low trade-size accounts on the same days,
while win rate stays flat across regimes. This suggests the edge on Fear days is a *sizing/volatility*
effect, not a *skill* effect. Rule of thumb: for traders already in the "high size" segment, allow a
modest (e.g. +20-30%) increase in position size specifically on Fear/Extreme-Fear days, capped by a hard
daily-loss stop, since realized variance (`pnl_std`) is also higher in Fear regimes (the tail risk cuts
both ways).

**Strategy idea 2 — Frequency throttle for low-size/infrequent traders during Greed.**
Infrequent and low-size traders do *not* show a Fear-day PnL edge (their averages are close to flat or
even lower on Fear days), meaning they are not capturing the same opportunity as the top segment. Rather
than increasing risk on Fear days for this group, the safer rule of thumb is: keep frequency and size
unchanged into Fear days for infrequent/low-size traders, and instead focus effort on trade *selection*
(higher win-rate entries) during Greed days, when the pooled long/short bias flips short and average
trade size for this segment stays roughly flat — i.e. this group's edge does not come from riding
sentiment momentum.

**Caveat:** none of the Fear vs Greed PnL differences clear a strict p<0.05 significance bar on this
sample (32 accounts, ~2 years), so these are *directional* rules of thumb to test further with more
data / a live paper-trading period, not settled statistical facts.


## Bonus — Predictive Model & Trader Clustering

In [ ]:
"""
Bonus:
1. Simple predictive model - predict next-day account profitability bucket
   (profit / loss) using today's sentiment + behavior features.
2. Cluster traders into behavioral archetypes (KMeans on account-level features).
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

daily_acct = pd.read_csv('outputs/daily_account_segmented.csv', parse_dates=['date'])
acct_summary = pd.read_csv('outputs/account_summary_segmented.csv')

# =================================================================
# 1. Predictive model: next-day profitable (1) vs not (0), per account
# =================================================================
df = daily_acct.sort_values(['Account', 'date']).copy()
df['is_profitable_day'] = (df['total_pnl'] > 0).astype(int)

# next day's outcome, within same account
df['next_day_profitable'] = df.groupby('Account')['is_profitable_day'].shift(-1)
df['sentiment_num'] = df['value']  # fear/greed index 0-100 for today

feature_cols = ['trades_count', 'total_pnl', 'win_rate', 'avg_trade_size_usd',
                 'total_volume_usd', 'long_short_ratio', 'sentiment_num']
model_df = df.dropna(subset=feature_cols + ['next_day_profitable']).copy()
model_df['long_short_ratio'] = model_df['long_short_ratio'].replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=['long_short_ratio'])

print("Model dataset shape:", model_df.shape)
print("Target balance:\n", model_df['next_day_profitable'].value_counts(normalize=True))

X = model_df[feature_cols]
y = model_df['next_day_profitable'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("\n=== Classification report (next-day profitable) ===")
print(classification_report(y_test, y_pred))
try:
    auc = roc_auc_score(y_test, y_proba)
    print("ROC AUC:", round(auc, 3))
except Exception as e:
    print("AUC error:", e)

importances = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nFeature importances:\n", importances)

fig, ax = plt.subplots(figsize=(7, 5))
importances.plot(kind='barh', ax=ax, color='#2c3e50')
ax.invert_yaxis()
ax.set_title('Feature Importance: Predicting Next-Day Profitability')
plt.tight_layout()
plt.savefig('outputs/chart7_feature_importance.png')
plt.close()

# =================================================================
# 2. Clustering traders into behavioral archetypes
# =================================================================
cluster_features = ['avg_trades_per_day', 'avg_trade_size_usd', 'avg_daily_pnl',
                     'avg_win_rate', 'win_rate_std']
cdf = acct_summary.dropna(subset=cluster_features).copy()

scaler = StandardScaler()
Xc = scaler.fit_transform(cdf[cluster_features])

k = 3
km = KMeans(n_clusters=k, random_state=42, n_init=10)
cdf['cluster'] = km.fit_predict(Xc)

print("\n=== Cluster profiles (mean of raw features) ===")
profile = cdf.groupby('cluster')[cluster_features].mean()
print(profile)
print("\nCluster sizes:\n", cdf['cluster'].value_counts())

cdf.to_csv('outputs/account_clusters.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(cdf['avg_trades_per_day'], cdf['avg_daily_pnl'],
                      c=cdf['cluster'], cmap='Set2', s=100, edgecolor='black')
ax.set_xlabel('Avg Trades / Day')
ax.set_ylabel('Avg Daily PnL (USD)')
ax.set_title('Trader Archetypes (KMeans, k=3)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.savefig('outputs/chart8_trader_clusters.png')
plt.close()

print("\nDONE - Bonus complete.")


### Chart 7 — Feature importance (next-day profitability model)
![chart7](../outputs/chart7_feature_importance.png)

### Chart 8 — Trader archetypes (KMeans, k=3)
![chart8](../outputs/chart8_trader_clusters.png)

### Bonus notes

- A Random Forest predicting **next-day account profitability** (profit vs loss) from today's behavior +
  sentiment reaches **ROC AUC ≈ 0.65** — meaningfully better than a coin flip but far from a tradeable
  signal on its own. The most important features are trade size, trade count, today's PnL, and the
  sentiment index value itself — confirming sentiment carries *some* predictive signal on top of pure
  trading behavior.
- **KMeans (k=3)** on account-level features (trade frequency, size, avg daily PnL, win rate, win-rate
  volatility) recovers three intuitive archetypes:
  - **Cluster 0 — "Volatile mid-size traders"**: moderate frequency/size, lower and noisier win rate.
  - **Cluster 1 — "Steady/consistent traders"**: lower frequency, high and stable win rate (~96%).
  - **Cluster 2 — "High-volume power traders"**: very high frequency and large trade size, strong win rate.


## Reproducibility

```bash
pip install -r requirements.txt
python scripts/01_prepare.py   # Part A: cleaning + metrics -> outputs/*.csv
python scripts/02_analysis.py  # Part B: sentiment analysis + segmentation -> outputs/chart1-6, csvs
python scripts/03_bonus.py     # Bonus: model + clustering -> outputs/chart7-8, csvs
```

All intermediate and final tables are saved under `outputs/`. See `README.md` for the full repo structure
and `WRITEUP.md` for the one-page methodology / insights / strategy summary.
